In [132]:
### RAG Pipeline- Data Ingestion to Vector DB Pipeline


In [133]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters pymupdf pypdf sentence-transformers

In [134]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
!pip install langchain-text-splitters


In [135]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
!pip install pypdf

In [136]:
def process_all_pdfs(pdf_directory):
    """Process all pdf files in the directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("data/pdf")

Found 3 PDF files to process

Processing: CGM Unit 5 Notes.pdf
Loaded 63 pages

Processing: CGM Unit 4 Notes.pdf
Loaded 48 pages

Processing: CGM Unit 6 Notes.pdf
Loaded 32 pages
Total documents loaded: 143


In [137]:
os.chdir("/Users/akshitasharma/RAG")
print(os.listdir("data"))

['text_file', 'pdf']


In [138]:
print(os.listdir("data/pdf"))

['.DS_Store', 'cgm pdf']


In [139]:
all_pdf_documents = process_all_pdfs("data/pdf")

Found 3 PDF files to process

Processing: CGM Unit 5 Notes.pdf
Loaded 63 pages

Processing: CGM Unit 4 Notes.pdf
Loaded 48 pages

Processing: CGM Unit 6 Notes.pdf
Loaded 32 pages
Total documents loaded: 143


In [140]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-04-20T10:55:45+00:00', 'source': 'data/pdf/cgm pdf/CGM Unit 5 Notes.pdf', 'total_pages': 63, 'page': 0, 'page_label': '1', 'source_file': 'CGM Unit 5 Notes.pdf', 'file_type': 'pdf'}, page_content='UNIT -5 COMPUTER GRAPHICS AND MULTIMEDIA\n1\nUNIT V MULTIMEDIA SYSTEMS DESIGN\nMultimedia basics − Multimedia applications − Multimedia system architecture −Evolving\ntechnologies for multimedia − Defining objects for multimedia systems −Multimedia data interface\nstandards −Compression And Decompression, Data And File Format Standards, Multimedia I/O\nTechnologies, Digital Voice And Audio, Video Image And Animation, Full Motion Video, Storage\nAnd Retrieval Technologies.\n3.1 Multimedia Basics\nMultimedia is a combination of text, graphic art, and sound, animation and video elements.\nThe IBM dictionary of computing describes multimedia as "comprehensive material, presented in a\ncombination 

In [141]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [142]:
chunks=split_documents(all_pdf_documents)

Split 143 documents into 367 chunks

Example chunk:
Content: UNIT -5 COMPUTER GRAPHICS AND MULTIMEDIA
1
UNIT V MULTIMEDIA SYSTEMS DESIGN
Multimedia basics − Multimedia applications − Multimedia system architecture −Evolving
technologies for multimedia − Definin...
Metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-04-20T10:55:45+00:00', 'source': 'data/pdf/cgm pdf/CGM Unit 5 Notes.pdf', 'total_pages': 63, 'page': 0, 'page_label': '1', 'source_file': 'CGM Unit 5 Notes.pdf', 'file_type': 'pdf'}


import numpy as np
pip install scikit-learn
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [143]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
            
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        

In [144]:
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 5459.81it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/s5/__ts1cy110d80_r4n5gsb2j40000gn/T/ipykernel_39121/1554396445.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [145]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any

In [146]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 367


In [147]:
texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 367 texts...


Batches: 100%|█████████████████████████████████| 12/12 [00:04<00:00,  2.59it/s]


Generated embeddings with shape: (367, 384)
Adding 367 documents to vector store...
Successfully added 367 documents to vector store
Total documents in collection: 734


In [148]:
##Retriever Pipeline From Vector Store

In [149]:
from typing import List, Dict, Any

In [150]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [151]:
rag_retriever

In [152]:
results = rag_retriever.retrieve("Hypermedia elements")
for doc in results:
    print(f"Rank: {doc['rank']}")
    print(f"Score: {doc['similarity_score']:.4f}")
    print(f"Content: {doc['content'][:200]}")
    print(f"Source: {doc['metadata'].get('source', 'unknown')}")
    print("---")

Retrieving documents for query: 'Hypermedia elements'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|███████████████████████████████████| 1/1 [00:00<00:00, 61.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
Rank: 1
Score: 0.3188
Content: 1.Hypermedia documents
Hypermedia documents are documents which have text, embedded or linked multimedia
objects such as image, audio, hologram, or full-motion video.The network speed and
computing ef
Source: data/pdf/cgm pdf/CGM Unit 5 Notes.pdf
---
Rank: 2
Score: 0.3188
Content: 1.Hypermedia documents
Hypermedia documents are documents which have text, embedded or linked multimedia
objects such as image, audio, hologram, or full-motion video.The network speed and
computing ef
Source: data/pdf/cgm pdf/CGM Unit 5 Notes.pdf
---
Rank: 3
Score: 0.2020
Content: interfaces such as Microsoft Windows.
The following figure describes the architecture of a multimedia workstation environment. In this
diagram.
The right side shows the new architectural entities requ
Source: data/pdf/cgm pdf/CGM Unit 5 Notes.pdf
---
Rank: 4
Score: 0.2020
Content: interfaces such as Microsoft Windows.
The

In [153]:
##RAG Pipeline- VectorDB To LLM Output Generation

In [154]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [155]:
import sys
!{sys.executable} -m pip install langchain-groq langchain-core langchain python-dotenv


In [156]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [157]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [158]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.3-70b-versatile
Groq LLM initialized successfully!


In [159]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Hypermedia")

Retrieving documents for query: 'Hypermedia'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|███████████████████████████████████| 1/1 [00:00<00:00, 42.79it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4e866b57_25',
  'content': '1.Hypermedia documents\nHypermedia documents are documents which have text, embedded or linked multimedia\nobjects such as image, audio, hologram, or full-motion video.The network speed and\ncomputing efficiency with which these hypermedia documents can manipulated has special\nimplications for multimedia application such as messaging. Hypermedia has its roots in\nhypertext.\nHypertext\nHypertext systems allow authors to link information together, create information paths through a\nlargevolume of related text in documents.It also allows to annotate existing text, and append notes.\nIt allows fast and easy searching and reading of selected excerpts.',
  'metadata': {'file_type': 'pdf',
   'total_pages': 63,
   'content_length': 648,
   'page': 5,
   'page_label': '6',
   'creationdate': '',
   'moddate': '2026-04-20T10:55:45+00:00',
   'creator': 'PyPDF',
   'doc_index': 25,
   'source_file': 'CGM Unit 5 Notes.pdf',
   'source': 'data/pdf/cgm pd

In [160]:
## Integration Vectordb Context pipeline With LLM outpput

In [161]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [162]:
answer=rag_simple("Hypermedia",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Hypermedia'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|███████████████████████████████████| 1/1 [00:00<00:00, 86.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Hypermedia refers to documents that contain text, embedded or linked multimedia objects such as images, audio, holograms, or full-motion video, with roots in hypertext.


In [163]:
## Enhanced RAG Pipeline Features

In [167]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question in detail.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output


# Example usage:
result = rag_advanced("What is multimedia", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)

print("=" * 60)
print("ANSWER:")
print(result['answer'])
print("\nCONFIDENCE:", round(result['confidence'], 4))
print("\nSOURCES:")
for s in result['sources']:
    print(f"  - {s['source']} | Page {s['page']} | Score: {round(s['score'], 4)}")
print("\nCONTEXT PREVIEW:")
print(result['context'][:300])
print("=" * 60)

Retrieving documents for query: 'What is multimedia'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|███████████████████████████████████| 1/1 [00:00<00:00, 32.03it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


ANSWER:
Multimedia refers to a combination of different forms of media, including text, graphic art, sound, animation, and video elements. It is a comprehensive material presented in a combination of these various forms, which can be used to convey information, entertain, or educate. According to the IBM dictionary of computing, multimedia is described as "comprehensive material, presented in a combination of text, graphics, video, animation and sound." Any system that is capable of presenting multimedia is called a multimedia system.

In essence, multimedia is an integrated approach to communication that incorporates multiple forms of media to engage users and convey information in a more interactive and immersive way. It can include a range of elements such as images, audio, video, animations, and text, which are combined to create a rich and engaging experience.

Multimedia applications can accept input from users through various means, including keyboards, voice commands, or pointi

In [169]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Video compression", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Video compression'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|███████████████████████████████████| 1/1 [00:00<00:00,  5.27it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Graphics and Multimedia Unit-5
35
Avideocaptureboardcanhandleavarietyofdifferentaudioandvideoinputsignalsandconvertthemfrom
analogtodigitalordigitaltoanalog.
VideoChannelMultiplexer:Itissimilartothevideograbber'svideochannelmultiplexer.
Video Compression and Decompression: A video compression and decompression processor is used to
compressanddecompressvideodata.
The video compression and decompression processor contains multiple stages for compression and
decompression. The stages include forward discrete cosine transformation and inverse discrete cosine
transformation, quantization and inverse quantization, ZigZag and Zero run-length encoding and decoding,
andmotionestimationandcompensation.
Audio Compression: MPEG-2 uses adaptive pulse code modulation (ADPCM) to sample the audio signal.
The method takes a difference b